# UNICEF Health & Socio-Economic Analysis
Analyzes UNICEF infant vaccination, breastfeeding, and socio-economic metadata using **Polars** (no Pandas) and **Plotnine** (ggplot-style) visualisations.

In [ ]:
!pip install polars plotnine geopandas adjustText

import polars as pl
from plotnine import *
import geopandas as gpd
import warnings
warnings.filterwarnings('ignore')

## 1. Data Loading & Transformation

In [ ]:
df_ind1 = pl.read_csv('unicef_indicator_1.csv', infer_schema_length=10000)
df_ind2 = pl.read_csv('unicef_indicator_2.csv', infer_schema_length=10000)
df_meta = pl.read_csv('unicef_metadata.csv', infer_schema_length=10000)

df_ind1 = (
    df_ind1.rename({'time_period': 'year', 'obs_value': 'polio_vaccine_pct'})
    .with_columns(pl.col('year').cast(pl.Int32, strict=False))
    .select(['country', 'alpha_3_code', 'year', 'polio_vaccine_pct'])
    .group_by(['country', 'alpha_3_code', 'year']).agg(pl.col('polio_vaccine_pct').mean())
)

df_ind2 = (
    df_ind2.rename({'time_period': 'year', 'obs_value': 'breastfed_pct'})
    .with_columns(pl.col('year').cast(pl.Int32, strict=False))
    .select(['alpha_3_code', 'year', 'breastfed_pct'])
    .group_by(['alpha_3_code', 'year']).agg(pl.col('breastfed_pct').mean())
)

meta_cols = {
    'alpha_3_code': 'alpha_3_code', 'year': 'year',
    'Population, total': 'population',
    'GDP per capita (constant 2015 US$)': 'gdp_per_capita',
    'Life expectancy at birth, total (years)': 'life_expectancy'
}
df_meta = (
    df_meta.select([pl.col(k).alias(v) for k, v in meta_cols.items()])
    .with_columns(pl.col('year').cast(pl.Int32, strict=False))
)

df_merged = df_ind1.join(df_ind2, on=['alpha_3_code', 'year'], how='outer')
df_merged = df_merged.join(df_meta, on=['alpha_3_code', 'year'], how='left')

TARGET_YEAR = 2019
df_TARGET = df_merged.filter(pl.col('year') == TARGET_YEAR)
df_TARGET.head()

## 2. World Map — Polio Vaccination Coverage

In [ ]:
df_map = df_TARGET.drop_nulls(subset='polio_vaccine_pct')
world = gpd.read_file('https://raw.githubusercontent.com/johan/world.geo.json/master/countries.geo.json')
polio_mapping = dict(zip(df_map['alpha_3_code'].to_list(), df_map['polio_vaccine_pct'].to_list()))
world['polio_vaccine_pct'] = world['id'].map(polio_mapping)

map_plot = (
    ggplot(world)
    + geom_map(aes(fill='polio_vaccine_pct'))
    + scale_fill_cmap('RdYlGn', na_value='#cccccc')
    + theme_minimal()
    + labs(title=f'Global Polio Vaccination Coverage ({TARGET_YEAR})', fill='Coverage (%)')
    + theme(figure_size=(11, 6))
)
map_plot.save('map_plot.png', width=11, height=6, dpi=300, verbose=False)
print('map_plot.png saved.')

## 3. Bar Chart — Bottom 10 Breastfeeding Rates (values at end of bars)

In [ ]:
bottom_10 = (
    df_TARGET.drop_nulls(subset=['country', 'breastfed_pct'])
    .sort('breastfed_pct', descending=False)
    .head(10)
)
bottom_10 = bottom_10.with_columns(
    pl.col('breastfed_pct').round(1).alias('breastfed_pct'),
    pl.col('breastfed_pct').round(1).cast(pl.Utf8).alias('value_label')
)
categories = bottom_10['country'].to_list()
bottom_10 = bottom_10.with_columns(pl.col('country').cast(pl.Enum(categories)))

bar_plot = (
    ggplot(bottom_10, aes(x='country', y='breastfed_pct', fill='breastfed_pct'))
    + geom_col(show_legend=True)
    + geom_text(aes(label='value_label'), ha='left', nudge_y=0.3, size=9, color='#333333')
    + scale_fill_cmap('RdYlGn')
    + coord_flip()
    + theme_minimal()
    + labs(title=f'Bottom 10 Countries by Breastfeeding Rate ({TARGET_YEAR})',
           x='Country', y='Breastfeeding Rate (%)', fill='Rate (%)')
    + theme(figure_size=(10, 6))
)
bar_plot.save('bar_plot.png', width=10, height=6, dpi=300, verbose=False)
print('bar_plot.png saved.')

## 4. Scatter Plot — GDP vs Life Expectancy with OLS + Bottom 10 Country Labels

In [ ]:
df_scatter = df_TARGET.drop_nulls(subset=['country', 'gdp_per_capita', 'life_expectancy', 'polio_vaccine_pct'])

# Identify bottom 10 countries by life expectancy to label
bottom10_countries = (
    df_scatter.sort('life_expectancy', descending=False)
    .head(10)['country'].to_list()
)

# Label column: name + value for bottom 10 only
df_scatter = df_scatter.with_columns(
    pl.when(pl.col('country').is_in(bottom10_countries))
    .then(pl.col('country') + '\n' + pl.col('life_expectancy').round(1).cast(pl.Utf8) + ' yrs')
    .otherwise(pl.lit(''))
    .alias('scatter_label')
)

scatter_plot = (
    ggplot(df_scatter, aes(x='gdp_per_capita', y='life_expectancy'))
    + geom_point(aes(color='polio_vaccine_pct'), alpha=0.75, size=3)
    + scale_color_cmap('viridis')
    + geom_smooth(method='lm', color='firebrick', fill='lightyellow', alpha=0.25)
    + geom_text(aes(label='scatter_label'), size=7, nudge_y=0.5,
                adjust_text={'expand_points': (1.5, 1.5)})
    + scale_x_log10()
    + theme_minimal()
    + labs(title=f'Life Expectancy vs GDP per Capita with OLS Trend ({TARGET_YEAR})',
           x='GDP per Capita (log scale, USD)', y='Life Expectancy (years)',
           color='Polio Coverage (%)')
    + theme(figure_size=(11, 7))
)
scatter_plot.save('scatter_plot.png', width=11, height=7, dpi=300, verbose=False)
print('scatter_plot.png saved.')

## 5. Time-Series — Global Vaccination Trend (Every Year on X-Axis)

In [ ]:
df_time = (
    df_merged.drop_nulls(subset=['year', 'polio_vaccine_pct'])
    .filter(pl.col('year').is_not_null())
    .group_by('year')
    .agg(pl.col('polio_vaccine_pct').mean().round(1))
    .sort('year')
)

min_yr = int(df_time['year'].min())
max_yr = int(df_time['year'].max())
# Every single integer year on x-axis
year_breaks = list(range(min_yr, max_yr + 1))

time_plot = (
    ggplot(df_time, aes(x='year', y='polio_vaccine_pct'))
    + geom_line(color='steelblue', size=1.2)
    + geom_point(color='#b22222', size=2.5)
    + scale_x_continuous(breaks=year_breaks, limits=[min_yr - 0.5, max_yr + 0.5])
    + theme_minimal()
    + labs(title='Global Average Polio Vaccination Coverage Over Time',
           x='Year', y='Average Coverage (%)')
    + theme(figure_size=(16, 5), axis_text_x=element_text(angle=90, ha='right', size=7))
)
time_plot.save('time_series_plot.png', width=16, height=5, dpi=300, verbose=False)
print('time_series_plot.png saved.')